In [2]:
from pathlib import Path

import altair as alt
import pandas as pd
from ecostyles import EcoStyles

styles = EcoStyles(); styles.register_and_enable_theme()

# ---------- data path: ../data first, else search up the folder tree ----------
FILENAME = "plot3_annual-balance.csv"
PATH = Path("../data") / FILENAME
if not PATH.exists():
    PATH = next(
        p for parent in [Path.cwd(), *list(Path.cwd().parents)[:4]]
        for p in parent.rglob(FILENAME)
    )
print("Using:", PATH.resolve())

# ---------- settings ----------
AGG = "mean"       # how the two half-years combine into a year: "mean" or "sum"
GREY = "#676A86"

# ---------- data ----------
raw = pd.read_csv(PATH)                                   # columns: period (e.g. 1991-S1), balance_pct_gdp
raw["year"] = raw["period"].str[:4].astype(int)
raw["balance"] = pd.to_numeric(raw["balance_pct_gdp"], errors="coerce")
raw = raw.dropna(subset=["balance"])

annual = raw.groupby("year")["balance"].agg(balance=AGG, halves="count").reset_index()
partial = annual.loc[annual["halves"] < 2, "year"].tolist()
annual = annual[annual["halves"] == 2].copy()             # keep complete years only
print("Dropped partial years:", partial)
annual["series"] = "Budget balance"                        # constant field so the theme assigns the colour

first, last = int(annual["year"].min()), int(annual["year"].max())
tick_years = [int(y) for y in annual["year"] if y % 5 == 0]

# ---------- chart ----------
y = alt.Y(
    "balance:Q", title="Budget balance, share of GDP",
    axis=alt.Axis(labelExpr="datum.label + '%'"),
)
bars = alt.Chart(annual).mark_bar().encode(
    x=alt.X("year:O", title=None, axis=alt.Axis(values=tick_years, labelAngle=0)),
    y=y,
    color=alt.Color("series:N", legend=None),
)
zero = alt.Chart(pd.DataFrame({"balance": [0]})).mark_rule(
    color=GREY, strokeWidth=0.8).encode(y=y)

caption = alt.Title(
    text="Source: Deutsche Bundesbank",
    subtitle=[f"Federal budget balance, % of GDP, {first}–{last}. "
              f"Annual figures are the {'mean' if AGG == 'mean' else 'sum'} of the two half-years."],
    orient="bottom", anchor="start",
    fontSize=11, subtitleFontSize=10,
    color=GREY, subtitleColor=GREY, dy=12)

chart = (
    (bars + zero)
    .properties(width=400, height=300)
    .configure(background="white", font="Circular Std")
    .configure_view(fill="transparent", stroke="transparent")
    .configure_axis(labelColor=GREY, titleColor=GREY))

styles.save(chart, path=".", name="germany_budget_balance", svg=True)
chart.save("germany_budget_balance.png", scale_factor=2.0)
chart

Using: /Users/g.rushworth/Documents/GitHub/RADataHub/Debt 2026/Peterson Blogs/Roger Schneider/data/plot3_annual-balance.csv
Dropped partial years: [2026]


alt.LayerChart(...)